# Modül 1 — Zihinsel Model: İnteraktif Deneyler

Bu notebook eğitimde anlattığımız kavramları canlı olarak denemenizi sağlıyor.

**Kurulum:**
```bash
pip install openai python-dotenv
```

Bir `.env` dosyası oluşturun:
```
OPENAI_API_KEY=sk-...
```

In [ ]:
# Kurulum ve yapılandırma
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def chat(prompt: str, system: str = '', temperature: float = 0.7, model: str = 'gpt-4o-mini') -> str:
    """Basit chat wrapper."""
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

print('Hazır!')

## Deney 1: Bağlamın Etkisi

Aynı görevi giderek zenginleşen bağlamla deneyelim.

In [ ]:
görev = "Lojistik şirketimizin müşteri hizmetleri için bir şikayet yanıt yazısı yaz."

# Versiyon 1: Sıfır bağlam
v1 = chat(görev)

# Versiyon 2: Kitle bağlamı eklendi
v2 = chat(görev + """

Bağlam: Müşteri 3 gün geciken kargosundan şikayetçi.
Ton: Samimi, özür dileyen ama savunmacı olmayan.
""")

# Versiyon 3: Tam 5 katman
v3 = chat("""
Rol: Kıdemli müşteri deneyimi uzmanısın.

Bağlam:
- Müşteri: 5 yıldır çalışıyoruz, VIP segment
- Sorun: Premium kargo 3 gün gecikmeli, iş toplantısı kaçırıldı
- Şirket sorumluluğu: Taşıyıcı hatası, bizim ihmalimiz değil ama müşteri bizi suçluyor
- Çözüm: Tam iade + %20 indirim kuponu onaylandı

Görev: Müşteriye yanıt e-postası yaz.

Kısıt:
- İlk paragraf sadece empati, özür — çözümden önce
- 'Maalesef' kelimesi yok
- Şirketi değil, deneyimi sahiplen
- 150 kelimeyi geçme

Format: Konu satırı + e-posta gövdesi
""")

print('=== VERSİYON 1: Sıfır bağlam ===')
print(v1)
print()
print('=== VERSİYON 2: Kısmi bağlam ===')
print(v2)
print()
print('=== VERSİYON 3: Tam 5 katman ===')
print(v3)

## Deney 2: Sıcaklık (Temperature) Etkisi

Aynı promptu farklı temperature değerlerinde çalıştırın.

In [ ]:
prompt = "Yapay zekanın iş hayatındaki etkisi hakkında 3 cümle yaz."

for temp in [0.0, 0.5, 1.0]:
    print(f'--- Temperature: {temp} ---')
    for i in range(2):  # Her temperature için 2 çalıştırma
        result = chat(prompt, temperature=temp)
        print(f'  [{i+1}] {result[:200]}...')
    print()

# Gözlem: Düşük temp → tutarlı, yüksek temp → çeşitli

## Deney 3: Rol Persona Etkisi

Aynı soru, farklı roller.

In [ ]:
soru = "Bir şirketin yapay zekayı benimsemesi için ilk adım nedir?"

roller = [
    ("Teknoloji şüphecisi CFO", "Sen teknoloji yatırımlarına şüpheyle bakan, ROI odaklı bir CFO'sun."),
    ("Hevesli CTO", "Sen her yeni teknolojiyi benimseyen, inovasyon odaklı bir CTO'sun."),
    ("Pragmatik danışman", "Sen onlarca şirketle dijital dönüşüm yapmış, hem başarıları hem başarısızlıkları gören bir danışmansın.")
]

for rol_adı, sistem_promptu in roller:
    cevap = chat(soru, system=sistem_promptu, temperature=0.3)
    print(f'=== {rol_adı} ===')
    print(cevap)
    print()

## Deney 4: Self-Consistency

Aynı analitik soruyu 5 kez sorun, tutarlılığı ölçün.

In [ ]:
analitik_soru = """
Bir SaaS şirketi 2 seçenekle karşı karşıya:
A) Ürüne yeni özellik ekle (3 ay geliştirme, gelir artışı belirsiz)
B) Mevcut özellikleri iyileştir (1 ay, churn azaltma beklentisi %15)

Hangisi daha iyi? Tek kelimeyle: A veya B.
Sonra 1 cümle gerekçe.
"""

sonuçlar = []
for i in range(5):
    cevap = chat(analitik_soru, temperature=0.7)
    sonuçlar.append(cevap)
    print(f'Çalıştırma {i+1}: {cevap[:150]}')

# Kaç kez A, kaç kez B?
a_count = sum(1 for c in sonuçlar if c.strip().upper().startswith('A'))
b_count = sum(1 for c in sonuçlar if c.strip().upper().startswith('B'))
print(f'\nSonuç: A={a_count}/5, B={b_count}/5')
print(f'Çoğunluk kararı: {"A" if a_count > b_count else "B"}')

## Deney 5: Halüsinasyon Tespiti

Modeli zor duruma sorun. Hangi sorularda uydurma yapıyor?

In [ ]:
# Bu prompt modeli 'emin olmadığını belirtmeye' zorluyor
güvenli_prompt_eki = """

Önemli: Emin olmadığın bilgiler için [?] işareti koy.
Bilmiyorsan 'bilmiyorum' de, uydurma.
"""

sorular = [
    "2024 yılında Türkiye'nin en büyük 5 AI startup'ı hangileri?",  # Belirsiz, halüsinasyon riski yüksek
    "Yapay zeka nedir?",  # Net, halüsinasyon riski düşük
    "Ahmet Yıldız adlı AI araştırmacısının son çalışması neydi?",  # Muhtemelen gerçek değil
]

for soru in sorular:
    print(f'SORU: {soru}')
    
    # Güvenli prompt eki olmadan
    cevap_normal = chat(soru, temperature=0.3)
    
    # Güvenli prompt eki ile
    cevap_güvenli = chat(soru + güvenli_prompt_eki, temperature=0.3)
    
    print(f'Normal: {cevap_normal[:200]}')
    print(f'Güvenli: {cevap_güvenli[:200]}')
    print()


## Deney 6: Kendi Promptunuzu Test Edin

Aşağıya kendi promptunuzu yazın ve 5 katman çerçevesini uygulayın.

In [ ]:
# ✏️ Buraya kendi promptunuzu yazın:
BENIM_PROMPTUM = """
YOUR PROMPT HERE
"""

# Mevcut promptu çalıştır
original = chat(BENIM_PROMPTUM)
print('=== ORİJİNAL ===')
print(original)
print()

# 5 katman analizi
analiz = chat(f"""
Aşağıdaki promptu 5 katman çerçevesiyle analiz et:

Prompt: {BENIM_PROMPTUM}

Her katman için:
- Mevcut mu? (Evet / Hayır / Kısmen)
- Varsa nasıl? (1 cümle)
- Eksikse ne eklenebilir? (1 cümle)

Katmanlar: Rol, Bağlam, Görev, Kısıt, Format

Son: Güçlendirilmiş versiyon (yeniden yaz)
""")

print('=== 5 KATMAN ANALİZİ ===')
print(analiz)
print()

# Güçlendirilmiş versiyonu çalıştır ve karşılaştır
# (analiz çıktısından güçlendirilmiş promptu kopyalayıp buraya yapıştırın)